<a href="https://colab.research.google.com/github/harnepal-hub/ai-trading-bot/blob/main/AI_trading_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ccxt pandas pandas-ta

In [ ]:
import requests
import pandas as pd
import pandas_ta as ta
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore') # Keeps the output clean

# --- CONFIGURATION ---
PAIR = "B-BTC_USDT"
INTERVAL = "1m"
LIMIT = 100
INITIAL_CAPITAL_USDT = 1200.00 # ~1 Lakh INR

def fetch_coindcx_data(pair, interval, limit):
    url = f"https://public.coindcx.com/market_data/candles?pair={pair}&interval={interval}&limit={limit}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        df = pd.DataFrame(data)
        df = df.sort_values(by='time')
        df.reset_index(drop=True, inplace=True)
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)

        # Ensure numbers are treated as decimals for math
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = df[col].astype(float)
        return df
    except Exception as e:
        print(f"Error fetching data: {e}")
        return None

def calculate_indicators(df):
    df.ta.vwap(append=True)
    df.ta.bbands(length=20, std=2, append=True)
    df.ta.rsi(length=14, append=True)
    df.ta.atr(length=14, append=True)
    df['volume_roc'] = df['volume'].pct_change(periods=3) * 100
    df.dropna(inplace=True)
    return df

class AIPaperBot:
    def __init__(self, initial_capital):
        self.balance = initial_capital
        self.position = None
        self.entry_price = 0
        self.sl = 0
        self.tp = 0
        self.position_size = 0
        self.total_trades = 0
        self.winning_trades = 0

        # 🧠 The AI's starting "DNA" (These will evolve automatically)
        self.vol_threshold = 200.0
        self.rsi_bullish = 50.0
        self.rsi_bearish = 50.0

    def evaluate(self, current_state, vwap_col):
        close = current_state['close']
        high = current_state['high']
        low = current_state['low']
        rsi = current_state['RSI_14']
        atr = current_state['ATRr_14']
        roc = current_state['volume_roc']
        vwap = current_state[vwap_col]
        current_time = current_state['datetime']

        # 1. If we are ALREADY in a trade, check for SL or TP limits
        if self.position is not None:
            if self.position == 'LONG':
                if low <= self.sl:
                    self.close_trade("Stop Loss Hit", self.sl, False, current_time)
                elif high >= self.tp:
                    self.close_trade("Take Profit Hit", self.tp, True, current_time)
            elif self.position == 'SHORT':
                if high >= self.sl:
                    self.close_trade("Stop Loss Hit", self.sl, False, current_time)
                elif low <= self.tp:
                    self.close_trade("Take Profit Hit", self.tp, True, current_time)

            if self.position is not None:
                # Still in trade, show floating PnL
                pnl = (close - self.entry_price) * self.position_size if self.position == 'LONG' else (self.entry_price - close) * self.position_size
                print(f"[{current_time}] ⏳ Holding {self.position} | Price: ${close:,.2f} | Open PnL: ${pnl:,.2f}")
            return

        # 2. If we are NOT in a trade, scan the market for entries
        print(f"[{current_time}] 👁️ Scanning | Price: ${close:,.2f} | Vol RoC: {roc:.1f}% (Req: >{self.vol_threshold:.1f}%) | RSI: {rsi:.1f}")

        if roc > self.vol_threshold:
            if close > vwap and rsi > self.rsi_bullish:
                self.open_trade('LONG', close, atr, current_time)
            elif close < vwap and rsi < self.rsi_bearish:
                self.open_trade('SHORT', close, atr, current_time)

    def open_trade(self, side, price, atr, time):
        self.position = side
        self.entry_price = price

        # Risk exactly 1% of current balance per trade
        risk_amount = self.balance * 0.01

        # Stop loss distance is dynamically set to 1.5x the current candle's volatility
        sl_distance = atr * 1.5
        self.position_size = risk_amount / sl_distance

        # Reward is 2x the risk (1:2 Risk/Reward Ratio)
        if side == 'LONG':
            self.sl = price - sl_distance
            self.tp = price + (sl_distance * 2)
        else:
            self.sl = price + sl_distance
            self.tp = price - (sl_distance * 2)

        print(f"\n🚀 [AI ACTION] ENTERING {side} TRADE")
        print(f"Time:  {time}")
        print(f"Entry: ${price:,.2f}")
        print(f"SL:    ${self.sl:,.2f} (Max Risk: ${risk_amount:,.2f})")
        print(f"TP:    ${self.tp:,.2f} (Target Reward: ${(risk_amount * 2):,.2f})")
        print(f"Size:  {self.position_size:.4f} BTC\n")

    def close_trade(self, reason, exit_price, is_win, time):
        self.total_trades += 1

        if self.position == 'LONG':
            gross_pnl = (exit_price - self.entry_price) * self.position_size
        else:
            gross_pnl = (self.entry_price - exit_price) * self.position_size

        # Deduct a simulated exchange fee (0.1% taker fee x2 for entering and exiting)
        fee = (exit_price * self.position_size) * 0.002
        net_pnl = gross_pnl - fee
        self.balance += net_pnl

        print(f"\n✅ [AI ACTION] EXITING TRADE - {reason}")
        print(f"Time:    {time}")
        print(f"Exit:    ${exit_price:,.2f}")
        print(f"Net PnL: ${net_pnl:,.2f} (Fees deducted: ${fee:,.2f})")
        print(f"Capital: ${self.balance:,.2f}\n")

        # 🧠 The Evolution Engine
        if is_win:
            self.winning_trades += 1
            print(f"🧠 AI EVOLVING: Trade won! Lowering volume threshold slightly to find more opportunities.")
            self.vol_threshold = max(100.0, self.vol_threshold - 10.0)
        else:
            print(f"🧠 AI EVOLVING: False signal. Trade lost. Increasing volume threshold to demand stronger momentum.")
            self.vol_threshold += 30.0 # Punish the AI heavily to avoid choppy markets

        print(f"🔄 New AI DNA -> Volume Threshold required for next trade: {self.vol_threshold:.1f}%\n")
        self.position = None

if __name__ == "__main__":
    print("==================================================")
    print("🤖 STARTING SELF-EVOLVING AI PAPER TRADER")
    print(f"💰 Initial Capital: ${INITIAL_CAPITAL_USDT:,.2f} USDT (~1 Lakh INR)")
    print("==================================================\n")

    bot = AIPaperBot(INITIAL_CAPITAL_USDT)

    while True:
        try:
            # Fetch the data
            raw_data = fetch_coindcx_data(PAIR, INTERVAL, LIMIT)

            if raw_data is not None and not raw_data.empty:
                processed_data = calculate_indicators(raw_data)

                # Get the very latest live tick
                current_state = processed_data.iloc[-1]
                vwap_col = [col for col in processed_data.columns if 'VWAP' in col][0]

                # Pass it to the AI Brain
                bot.evaluate(current_state, vwap_col)

            # Wait 30 seconds before polling the market again so the loop runs continuously
            time.sleep(30)

        except KeyboardInterrupt:
            # If you press the "Stop" button in Colab, it prints a final report
            print("\n🛑 AI Bot Stopped by User.")
            print(f"📊 Final Balance: ${bot.balance:,.2f}")
            print(f"🏆 Win Rate: {(bot.winning_trades / max(1, bot.total_trades)) * 100:.1f}%")
            break
        except Exception as e:
            print(f"⚠️ Loop Error: {e}")
            time.sleep(30)

🤖 STARTING SELF-EVOLVING AI PAPER TRADER
💰 Initial Capital: $1,200.00 USDT (~1 Lakh INR)

[2026-09-12 14:12:00] 👁️ Scanning | Price: $77,368.95 | Vol RoC: 7.9% (Req: >200.0%) | RSI: 66.1
[2026-09-12 14:12:00] 👁️ Scanning | Price: $77,368.95 | Vol RoC: 78.7% (Req: >200.0%) | RSI: 66.1
[2026-09-12 14:13:00] 👁️ Scanning | Price: $77,368.94 | Vol RoC: -97.6% (Req: >200.0%) | RSI: 66.1
[2026-09-12 14:13:00] 👁️ Scanning | Price: $77,368.95 | Vol RoC: -95.1% (Req: >200.0%) | RSI: 66.1
